# 04 — SHAP Explainability

**Purpose:** Open the black box. Understand *why* the model makes specific predictions.

**Run after notebook 03.** Assumes `model`, `X_test`, `y_test`, and `FEATURE_COLS` are in memory.

**What is SHAP?**
SHAP (SHapley Additive exPlanations) is a method from game theory applied to ML. For any single prediction, it asks: *"how much did each feature contribute to this prediction compared to the average?"*

Example output for one product:
```
Predicted: 18 units  |  Base (average): 7.3 units
  lag_1_qty            +6.2   (sold 22 units yesterday → model expects more)
  day_of_week          +3.1   (Mondays historically strong for this product)
  last_7_day_avg       +2.8   (rising short-term trend)
  is_holiday           -1.4   (holiday → lower demand)
```

This lets you verify the model is reasoning correctly — not getting lucky with wrong logic.

---

## Cell 1 — Imports and sample

Computing SHAP on the full test set (millions of rows) would take hours. We use a **random 5000-row sample** — enough to get reliable feature importance estimates.

**`TreeExplainer`** is SHAP's fast implementation for tree-based models like XGBoost. It's exact (not approximate) and runs in seconds on 5000 rows.

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

# Sample 5000 rows from test set for SHAP
SAMPLE_SIZE = 5000
X_sample = X_test.sample(SAMPLE_SIZE, random_state=42)
y_sample = y_test.loc[X_sample.index]

print(f"SHAP sample: {len(X_sample):,} rows")
print("Columns:", list(X_sample.columns))

## Cell 2 — Compute SHAP values

For each of the 5000 rows, SHAP computes one value per feature: the contribution of that feature to the prediction.

**Output shape:** `(5000, 12)` — one SHAP value per feature per row.

**Positive SHAP** = feature pushed prediction *up* (e.g., high lag_1_qty → predict more tomorrow)

**Negative SHAP** = feature pushed prediction *down* (e.g., is_holiday=True → predict less tomorrow)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected output columns: {len(FEATURE_COLS)} features")

## Cell 3 — Summary bar chart: mean |SHAP| per feature

This answers: *"which features matter most overall?"*

**Mean absolute SHAP** = average impact on prediction magnitude across all samples. Higher = more important.

**Expected ranking (approximately):**
1. `lag_1_qty` — yesterday's sales is almost always the strongest predictor
2. `last_7_day_avg` — short-term trend
3. `product_id_encoded` — per-product bias
4. `horizon_days` — model adjusts for how far ahead it's predicting
5. `day_of_week` — weekday patterns

If something unexpected ranks first (e.g., `is_holiday` on a dataset where calendar_dim was never seeded), investigate — it may indicate a data issue.

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=True)

print(importance_df.sort_values("mean_abs_shap", ascending=False).to_string(index=False))

importance_df.plot.barh(x="feature", y="mean_abs_shap", figsize=(8, 5), legend=False, color="steelblue")
plt.xlabel("Mean |SHAP value|")
plt.title("Feature importance (SHAP)")
plt.tight_layout()
plt.show()

## Cell 4 — Beeswarm plot: direction and magnitude

The beeswarm plot shows **both** how important a feature is AND whether high/low values push predictions up or down.

**How to read it:**
- Each dot = one prediction from the sample
- X position = SHAP value (positive = pushed prediction up, negative = down)
- Color = feature value (red = high, blue = low)

**What to look for:**
- `lag_1_qty`: red dots (high yesterday's sales) should be on the right (positive SHAP) → model correctly learns "high yesterday = predict high today"
- `is_holiday`: if present, blue dots (no holiday) should have near-zero SHAP, colored dots should vary

In [ ]:
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, max_display=12)

## Cell 5 — Waterfall for one product

A waterfall chart shows exactly how one specific prediction was built up from the baseline.

**Useful for business conversations:** You can say *"for product X on Monday, the model predicted 18 units because it sold 22 yesterday (+6.2), and Mondays are historically strong (+3.1), but tomorrow is a holiday (-1.4)."*

In [ ]:
# Pick one row (e.g., a product with high predicted demand)
sample_idx = X_sample.index[0]

explanation = shap.Explanation(
    values=shap_values[0],
    base_values=explainer.expected_value,
    data=X_sample.iloc[0].values,
    feature_names=FEATURE_COLS,
)
shap.plots.waterfall(explanation)

print(f"\nActual qty sold: {y_sample.iloc[0]:.1f}")
print(f"Model predicted: {model.predict(X_sample.iloc[[0]])[0]:.2f}")

## Cell 6 — Save SHAP importance for pipeline use

The production `train.py` script computes SHAP importance and logs it to MLflow automatically. This cell shows what that output looks like — use it to verify the format matches what `train.py` expects.

In [ ]:
import json

shap_importance = [
    {"feature": feat, "mean_abs_shap": round(float(np.abs(shap_values[:, i]).mean()), 4)}
    for i, feat in enumerate(FEATURE_COLS)
]
shap_importance.sort(key=lambda x: x["mean_abs_shap"], reverse=True)

print("SHAP importance JSON (top 5):")
print(json.dumps(shap_importance[:5], indent=2))

## Cell 7 — Final sanity check before handing off to production

Before we move to `train.py`, verify:
1. No negative predictions on test set (XGBoost can produce small negatives for near-zero demand products)
2. Coverage — predictions exist for all products in test set
3. Outliers — how many predictions are absurdly high (> 3× product max historical)?

These checks translate directly into the sanity logic in `predict.py`.

In [ ]:
test_preds = model.predict(X_test)

n_negative = (test_preds < 0).sum()
print(f"Negative predictions: {n_negative} ({n_negative/len(test_preds)*100:.3f}%)")
print(f"Min prediction: {test_preds.min():.4f}")
print(f"Max prediction: {test_preds.max():.1f}")
print(f"\nPredictions > 100 units: {(test_preds > 100).sum():,}")
print(f"Predictions > 500 units: {(test_preds > 500).sum():,}")